# Friedman Test — Accuracy & G-Mean

Non-parametric test on 8 classifiers across **16 evaluation blocks**
(4 UCI credit datasets × 4 feature-selection variants per dataset).

| Metric | Source | Blocks |
|---|---|---|
| **Accuracy** | `<dataset>_v3_full_comparison.csv` | 16 (4 × 4 FS) |
| **G-Mean**   | `<dataset>_v3_ten_measure_results.csv` | 16 (4 × 4 FS) |

Ranks 1 = best, 8 = worst per block; ties take average ranks.

- $H_0$: all 8 classifiers perform equally on these data.
- $H_1$: at least one differs.

Friedman $\chi^2$ + Iman–Davenport F-correction (less conservative).
> Embedded data — open in Colab and Run All. No uploads.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, chi2, f as f_dist


## 2. Embedded scores (4 datasets × 4 FS × 8 models)

In [2]:
MODELS = ['LogisticReg', 'KNN', 'SVM-RBF', 'DecisionTree', 'RandomForest', 'GradBoost', 'XGBoost', 'Stacker']
FS_ORDER = ['Full', 'RFE', 'Lasso-LR', 'Forward']

# Accuracy: 16 blocks  (§6 leaderboard)
ACC = {
    'Australian': {
        'Full': {'LogisticReg': 0.8333, 'KNN': 0.8551, 'SVM-RBF': 0.8333, 'DecisionTree': 0.8116, 'RandomForest': 0.8188, 'GradBoost': 0.8551, 'XGBoost': 0.8696, 'Stacker': 0.8623},
        'RFE': {'LogisticReg': 0.8333, 'KNN': 0.8261, 'SVM-RBF': 0.8333, 'DecisionTree': 0.7971, 'RandomForest': 0.8188, 'GradBoost': 0.8333, 'XGBoost': 0.8333, 'Stacker': 0.8333},
        'Lasso-LR': {'LogisticReg': 0.8333, 'KNN': 0.8043, 'SVM-RBF': 0.8333, 'DecisionTree': 0.8261, 'RandomForest': 0.8116, 'GradBoost': 0.8478, 'XGBoost': 0.8768, 'Stacker': 0.8406},
        'Forward': {'LogisticReg': 0.7971, 'KNN': 0.8116, 'SVM-RBF': 0.7826, 'DecisionTree': 0.8261, 'RandomForest': 0.8188, 'GradBoost': 0.8188, 'XGBoost': 0.8116, 'Stacker': 0.8116},
    },
    'Japanese': {
        'Full': {'LogisticReg': 0.8623, 'KNN': 0.8116, 'SVM-RBF': 0.8478, 'DecisionTree': 0.8841, 'RandomForest': 0.8768, 'GradBoost': 0.9058, 'XGBoost': 0.8986, 'Stacker': 0.8986},
        'RFE': {'LogisticReg': 0.8841, 'KNN': 0.8696, 'SVM-RBF': 0.7971, 'DecisionTree': 0.8768, 'RandomForest': 0.8986, 'GradBoost': 0.9058, 'XGBoost': 0.8768, 'Stacker': 0.9275},
        'Lasso-LR': {'LogisticReg': 0.8551, 'KNN': 0.8261, 'SVM-RBF': 0.8406, 'DecisionTree': 0.8261, 'RandomForest': 0.9130, 'GradBoost': 0.8913, 'XGBoost': 0.9203, 'Stacker': 0.8986},
        'Forward': {'LogisticReg': 0.8913, 'KNN': 0.8478, 'SVM-RBF': 0.8841, 'DecisionTree': 0.8261, 'RandomForest': 0.8841, 'GradBoost': 0.8913, 'XGBoost': 0.8913, 'Stacker': 0.9058},
    },
    'German': {
        'Full': {'LogisticReg': 0.6850, 'KNN': 0.6450, 'SVM-RBF': 0.7400, 'DecisionTree': 0.5850, 'RandomForest': 0.7100, 'GradBoost': 0.7450, 'XGBoost': 0.6750, 'Stacker': 0.7100},
        'RFE': {'LogisticReg': 0.6900, 'KNN': 0.6950, 'SVM-RBF': 0.7500, 'DecisionTree': 0.7250, 'RandomForest': 0.6900, 'GradBoost': 0.7700, 'XGBoost': 0.7500, 'Stacker': 0.7400},
        'Lasso-LR': {'LogisticReg': 0.7150, 'KNN': 0.6700, 'SVM-RBF': 0.7150, 'DecisionTree': 0.6800, 'RandomForest': 0.7100, 'GradBoost': 0.7350, 'XGBoost': 0.6950, 'Stacker': 0.7150},
        'Forward': {'LogisticReg': 0.7100, 'KNN': 0.7000, 'SVM-RBF': 0.7350, 'DecisionTree': 0.6350, 'RandomForest': 0.6650, 'GradBoost': 0.7100, 'XGBoost': 0.6800, 'Stacker': 0.7000},
    },
    'Taiwan': {
        'Full': {'LogisticReg': 0.8048, 'KNN': 0.7750, 'SVM-RBF': 0.8013, 'DecisionTree': 0.7973, 'RandomForest': 0.8003, 'GradBoost': 0.7962, 'XGBoost': 0.8007, 'Stacker': 0.8003},
        'RFE': {'LogisticReg': 0.7913, 'KNN': 0.7773, 'SVM-RBF': 0.7950, 'DecisionTree': 0.8087, 'RandomForest': 0.7698, 'GradBoost': 0.7967, 'XGBoost': 0.7970, 'Stacker': 0.7933},
        'Lasso-LR': {'LogisticReg': 0.8053, 'KNN': 0.7733, 'SVM-RBF': 0.8028, 'DecisionTree': 0.8057, 'RandomForest': 0.8025, 'GradBoost': 0.7902, 'XGBoost': 0.7988, 'Stacker': 0.7983},
        'Forward': {'LogisticReg': 0.7858, 'KNN': 0.7918, 'SVM-RBF': 0.7892, 'DecisionTree': 0.7840, 'RandomForest': 0.8053, 'GradBoost': 0.7888, 'XGBoost': 0.8023, 'Stacker': 0.8012},
    },
}

# G-Mean: 16 blocks  (§7 ten-measure suite)
GMEAN = {
    'Australian': {
        'Full': {'LogisticReg': 0.8390, 'KNN': 0.8508, 'SVM-RBF': 0.8364, 'DecisionTree': 0.8087, 'RandomForest': 0.8241, 'GradBoost': 0.8598, 'XGBoost': 0.8724, 'Stacker': 0.8645},
        'RFE': {'LogisticReg': 0.8390, 'KNN': 0.8305, 'SVM-RBF': 0.8387, 'DecisionTree': 0.8013, 'RandomForest': 0.8236, 'GradBoost': 0.8382, 'XGBoost': 0.8374, 'Stacker': 0.8390},
        'Lasso-LR': {'LogisticReg': 0.8390, 'KNN': 0.8042, 'SVM-RBF': 0.8374, 'DecisionTree': 0.8271, 'RandomForest': 0.8170, 'GradBoost': 0.8521, 'XGBoost': 0.8803, 'Stacker': 0.8452},
        'Forward': {'LogisticReg': 0.8024, 'KNN': 0.8171, 'SVM-RBF': 0.7874, 'DecisionTree': 0.8316, 'RandomForest': 0.8241, 'GradBoost': 0.8218, 'XGBoost': 0.8159, 'Stacker': 0.8170},
    },
    'German': {
        'Full': {'LogisticReg': 0.6137, 'KNN': 0.5294, 'SVM-RBF': 0.6492, 'DecisionTree': 0.6298, 'RandomForest': 0.6882, 'GradBoost': 0.6170, 'XGBoost': 0.6330, 'Stacker': 0.6711},
        'RFE': {'LogisticReg': 0.6708, 'KNN': 0.6083, 'SVM-RBF': 0.6705, 'DecisionTree': 0.6835, 'RandomForest': 0.6581, 'GradBoost': 0.6294, 'XGBoost': 0.6990, 'Stacker': 0.6992},
        'Lasso-LR': {'LogisticReg': 0.6429, 'KNN': 0.5606, 'SVM-RBF': 0.6604, 'DecisionTree': 0.6294, 'RandomForest': 0.6544, 'GradBoost': 0.5928, 'XGBoost': 0.6583, 'Stacker': 0.6937},
        'Forward': {'LogisticReg': 0.6473, 'KNN': 0.6414, 'SVM-RBF': 0.6619, 'DecisionTree': 0.6245, 'RandomForest': 0.6448, 'GradBoost': 0.6882, 'XGBoost': 0.6234, 'Stacker': 0.6848},
    },
    'Japanese': {
        'Full': {'LogisticReg': 0.8612, 'KNN': 0.7785, 'SVM-RBF': 0.8445, 'DecisionTree': 0.8871, 'RandomForest': 0.8760, 'GradBoost': 0.8990, 'XGBoost': 0.8951, 'Stacker': 0.8951},
        'RFE': {'LogisticReg': 0.8857, 'KNN': 0.8513, 'SVM-RBF': 0.7863, 'DecisionTree': 0.8803, 'RandomForest': 0.8971, 'GradBoost': 0.9014, 'XGBoost': 0.8760, 'Stacker': 0.9224},
        'Lasso-LR': {'LogisticReg': 0.8548, 'KNN': 0.7983, 'SVM-RBF': 0.8382, 'DecisionTree': 0.8235, 'RandomForest': 0.9051, 'GradBoost': 0.8842, 'XGBoost': 0.9161, 'Stacker': 0.8875},
        'Forward': {'LogisticReg': 0.8923, 'KNN': 0.8445, 'SVM-RBF': 0.8842, 'DecisionTree': 0.7983, 'RandomForest': 0.8824, 'GradBoost': 0.8842, 'XGBoost': 0.8907, 'Stacker': 0.9084},
    },
    'Taiwan': {
        'Full': {'LogisticReg': 0.6257, 'KNN': 0.6399, 'SVM-RBF': 0.6363, 'DecisionTree': 0.6358, 'RandomForest': 0.6703, 'GradBoost': 0.6719, 'XGBoost': 0.6662, 'Stacker': 0.6789},
        'RFE': {'LogisticReg': 0.6143, 'KNN': 0.6167, 'SVM-RBF': 0.6153, 'DecisionTree': 0.6088, 'RandomForest': 0.6751, 'GradBoost': 0.6258, 'XGBoost': 0.6406, 'Stacker': 0.6482},
        'Lasso-LR': {'LogisticReg': 0.6254, 'KNN': 0.6446, 'SVM-RBF': 0.6453, 'DecisionTree': 0.6317, 'RandomForest': 0.6564, 'GradBoost': 0.6683, 'XGBoost': 0.6691, 'Stacker': 0.6801},
        'Forward': {'LogisticReg': 0.6468, 'KNN': 0.6378, 'SVM-RBF': 0.6401, 'DecisionTree': 0.6468, 'RandomForest': 0.6510, 'GradBoost': 0.6207, 'XGBoost': 0.6222, 'Stacker': 0.6362},
    },
}

print('Accuracy blocks:', len(ACC) * len(FS_ORDER))
print('G-Mean blocks  :', len(GMEAN) * len(FS_ORDER))


Accuracy blocks: 16
G-Mean blocks  : 16


## 3. Build score matrices (16 × 8 each)

In [3]:
def build_matrix(score_dict):
    rows, index = [], []
    for ds in score_dict:
        for fs in FS_ORDER:
            index.append(f'{ds} | {fs}')
            rows.append([score_dict[ds][fs][m] for m in MODELS])
    return pd.DataFrame(rows, index=index, columns=MODELS)

mat_acc   = build_matrix(ACC)
mat_gmean = build_matrix(GMEAN)
print('Accuracy matrix:', mat_acc.shape)
print(mat_acc.round(4))
print()
print('G-Mean matrix:', mat_gmean.shape)
print(mat_gmean.round(4))


Accuracy matrix: (16, 8)
                       LogisticReg     KNN  SVM-RBF  DecisionTree  \
Australian | Full           0.8333  0.8551   0.8333        0.8116   
Australian | RFE            0.8333  0.8261   0.8333        0.7971   
Australian | Lasso-LR       0.8333  0.8043   0.8333        0.8261   
Australian | Forward        0.7971  0.8116   0.7826        0.8261   
Japanese | Full             0.8623  0.8116   0.8478        0.8841   
Japanese | RFE              0.8841  0.8696   0.7971        0.8768   
Japanese | Lasso-LR         0.8551  0.8261   0.8406        0.8261   
Japanese | Forward          0.8913  0.8478   0.8841        0.8261   
German | Full               0.6850  0.6450   0.7400        0.5850   
German | RFE                0.6900  0.6950   0.7500        0.7250   
German | Lasso-LR           0.7150  0.6700   0.7150        0.6800   
German | Forward            0.7100  0.7000   0.7350        0.6350   
Taiwan | Full               0.8048  0.7750   0.8013        0.7973   
Taiwan | 

## 4. Per-block ranks (1 = best, 8 = worst)

In [4]:
def ranks(mat):
    return mat.rank(axis=1, method='average', ascending=False)

rank_acc   = ranks(mat_acc)
rank_gmean = ranks(mat_gmean)

print('Average ranks on ACCURACY (lower = better):')
print(rank_acc.mean().sort_values().round(3))
print()
print('Average ranks on G-MEAN (lower = better):')
print(rank_gmean.mean().sort_values().round(3))


Average ranks on ACCURACY (lower = better):
GradBoost       3.094
Stacker         3.375
XGBoost         3.406
SVM-RBF         4.375
LogisticReg     4.500
RandomForest    4.969
DecisionTree    5.750
KNN             6.531
dtype: float64

Average ranks on G-MEAN (lower = better):
Stacker         2.188
XGBoost         3.656
GradBoost       3.844
RandomForest    4.000
LogisticReg     5.000
SVM-RBF         5.094
DecisionTree    5.781
KNN             6.438
dtype: float64


## 5. Friedman + Iman–Davenport

In [5]:
def friedman_id(mat, alpha=0.05):
    n, k = mat.shape
    chi2_F, p_chi = friedmanchisquare(*[mat[m].values for m in MODELS])
    dof_chi = k - 1
    F_F   = ((n - 1) * chi2_F) / (n * (k - 1) - chi2_F)
    dof_F = (k - 1, (k - 1) * (n - 1))
    p_F   = 1 - f_dist.cdf(F_F, dof_F[0], dof_F[1])
    crit_chi = chi2.ppf(1 - alpha, dof_chi)
    crit_F   = f_dist.ppf(1 - alpha, dof_F[0], dof_F[1])
    return dict(n=n, k=k, chi2=chi2_F, p_chi=p_chi, dof_chi=dof_chi, crit_chi=crit_chi,
                F=F_F, p_F=p_F, dof_F=dof_F, crit_F=crit_F)

for name, mat in [('ACCURACY', mat_acc), ('G-MEAN', mat_gmean)]:
    r = friedman_id(mat)
    print(f'\n=== {name} ===')
    print(f"  Blocks n = {r['n']}, classifiers k = {r['k']}")
    print(f"  Friedman   chi^2 = {r['chi2']:.3f}  dof = {r['dof_chi']}  p = {r['p_chi']:.5f}  crit(0.05) = {r['crit_chi']:.3f}  -> H0 {'REJECTED' if r['p_chi']<0.05 else 'not rejected'}")
    print(f"  Iman-Dav.  F    = {r['F']:.3f}  dof = {r['dof_F']}  p = {r['p_F']:.5f}  crit(0.05) = {r['crit_F']:.3f}  -> H0 {'REJECTED' if r['p_F']<0.05 else 'not rejected'}")



=== ACCURACY ===
  Blocks n = 16, classifiers k = 8
  Friedman   chi^2 = 28.615  dof = 7  p = 0.00017  crit(0.05) = 14.067  -> H0 REJECTED
  Iman-Dav.  F    = 5.147  dof = (7, 105)  p = 0.00005  crit(0.05) = 2.098  -> H0 REJECTED

=== G-MEAN ===
  Blocks n = 16, classifiers k = 8
  Friedman   chi^2 = 34.070  dof = 7  p = 0.00002  crit(0.05) = 14.067  -> H0 REJECTED
  Iman-Dav.  F    = 6.558  dof = (7, 105)  p = 0.00000  crit(0.05) = 2.098  -> H0 REJECTED


## 6. Summary table

In [6]:
rows = []
for m in MODELS:
    rows.append({'Model': m,
                 'Accuracy rank': round(rank_acc[m].mean(), 2),
                 'G-Mean rank':   round(rank_gmean[m].mean(), 2)})
summary = pd.DataFrame(rows).sort_values('G-Mean rank').reset_index(drop=True)
print('Average ranks (lower = better):')
print(summary.to_string(index=False))


Average ranks (lower = better):
       Model  Accuracy rank  G-Mean rank
     Stacker           3.38         2.19
     XGBoost           3.41         3.66
   GradBoost           3.09         3.84
RandomForest           4.97         4.00
 LogisticReg           4.50         5.00
     SVM-RBF           4.38         5.09
DecisionTree           5.75         5.78
         KNN           6.53         6.44


## 7. Interpretation

If Friedman rejects $H_0$ → the 8 classifiers are not equivalent on this data.
The classifier with the lowest average rank is the best across the blocks.

If both Accuracy and G-Mean reject, the conclusion is robust to which metric is used.